# Qwen technique probe: killing deliberation (run 023)
Serves the model (proven cells), then tries 3 anti-deliberation
techniques x 2 cases and saves `/kaggle/working/probes.json`.
Settings: GPU T4 x2, Internet ON. ~25 min total.


In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
print(result.stdout)
assert result.returncode == 0, 'No GPU visible — enable a GPU accelerator in notebook settings (T4 x2 recommended).'
gpus = [line for line in result.stdout.strip().split('\n') if line.strip()]
print(f'detected {len(gpus)} GPU(s)')
print('--- driver / CUDA version (diagnostic) ---')
drv = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('\n'.join(drv.stdout.split('\n')[:11]))


In [ ]:
REPO = 'ISTA-DASLab/Qwen3.8-27B-GSQ-RCO-GGUF'
GGUF_FILE = 'Qwen3.8-27B-GSQ-RCO-IQ3_S.gguf'  # ~12 GB. Alternatives in README.
N_CTX = 8192
N_GPU_LAYERS = -1  # -1 = offload all; lower (e.g. 40) on CUDA OOM.
PORT = 8080
print('config OK:', REPO, GGUF_FILE)


In [ ]:
import glob
import os
import site
import subprocess
import sys


def run(cmd, check=True):
    print('+', ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=check, capture_output=True, text=True)


def pip_install(*args):
    run([sys.executable, '-m', 'pip', 'install', '-q', *args])


pip_install('--upgrade', 'pip')
pip_install(
    'llama-cpp-python', '--extra-index-url',
    'https://abetlen.github.io/llama-cpp-python/whl/cu124',
)
pip_install('huggingface_hub', 'hf_transfer')
# Server runtime deps (llama_cpp.server needs all of these; v3 died on sse_starlette).
# Server runtime closure: upstream extra first, explicit backstop second.
# (v4 died on starlette_context; hand-enumeration misses lazy imports.)
pip_install('llama-cpp-python[server]', '--extra-index-url',
            'https://abetlen.github.io/llama-cpp-python/whl/cu124')
pip_install('sse-starlette', 'starlette-context', 'fastapi',
            'uvicorn[standard]', 'pydantic', 'pydantic-settings',
            'anyio', 'httpx')
pip_install('nvidia-cuda-runtime-cu12==12.4.*', 'nvidia-cublas-cu12==12.4.*')
sp = site.getsitepackages()[0]
libs = [f'{sp}/nvidia/cuda_runtime/lib', f'{sp}/nvidia/cublas/lib']
for d in libs:
    out = run(['ls', d])
    print(d, '->', out.stdout.strip().split(chr(10))[:6])
    assert 'libcudart' in out.stdout or 'libcublas' in out.stdout, f'missing libs in {d}'
os.environ['LD_LIBRARY_PATH'] = ':'.join(libs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('LD_LIBRARY_PATH:', os.environ['LD_LIBRARY_PATH'])
so = glob.glob(f'{sp}/llama_cpp/lib/libllama.so*')
assert so, "libllama.so not found under site-packages/llama_cpp/lib"
print('--- ldd libllama (want: no not found) ---')
print(run(['ldd', so[0]], check=False).stdout[-1500:])
print('--- registering pip CUDA libs system-wide ---')
subprocess.run(['sudo', '-n', 'tee', '/etc/ld.so.conf.d/nvidia-pip.conf'],
                 input='\n'.join(libs) + '\n', check=True,
                 capture_output=True, text=True)
subprocess.run(['sudo', '-n', 'ldconfig'], check=True, capture_output=True, text=True)
print(subprocess.run('ldconfig -p | grep -m6 cudart', shell=True, capture_output=True, text=True).stdout)
# Verify the actually-used module (v3 only checked the base import).
import llama_cpp.server
print('llama_cpp.server OK (imports clean)')


In [ ]:
import os
import time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

t0 = time.time()
GGUF_PATH = hf_hub_download(repo_id=REPO, filename=GGUF_FILE)
print(f'downloaded in {(time.time() - t0) / 60:.1f} min:', GGUF_PATH)


In [ ]:
import json
import subprocess
import time
import urllib.request

log = open('/tmp/llama-server.log', 'w')
cmd = [
    'python', '-m', 'llama_cpp.server',
    '--model', GGUF_PATH,
    '--n_gpu_layers', str(N_GPU_LAYERS),
    '--n_ctx', str(N_CTX),
    '--host', '127.0.0.1',
    '--port', str(PORT),
]
print('launch:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
for i in range(60):
    if proc.poll() is not None:
        print('--- server died on startup; log tail ---')
        print(''.join(open('/tmp/llama-server.log').readlines()[-40:]))
        raise RuntimeError(f"server exited with code {proc.returncode}")
    time.sleep(10)
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/v1/models', timeout=5) as r:
            print('READY, status', r.status)
            break
    except Exception:
        if i % 6 == 0:
            print(f'waiting for server... ({i * 10}s)')
else:
    print('--- server log tail ---')
    print(''.join(open('/tmp/llama-server.log').readlines()[-30:]))
    raise RuntimeError('server did not come up in 10 min')


In [ ]:
import json
import urllib.request


def chat(content, max_tokens=50):
    body = json.dumps({
        'messages': [{'role': 'user', 'content': content}],
        'temperature': 0,
        'max_tokens': max_tokens,
    }).encode()
    req = urllib.request.Request(
        f'http://127.0.0.1:{PORT}/v1/chat/completions',
        data=body,
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)


d = chat('Reply with exactly: ok')
text = d['choices'][0]['message']['content']
print('reply:', repr(text[:120]))
assert text and text.strip(), 'empty reply — model misbehaving; check /tmp/llama-server.log'
assert d['choices'][0]['finish_reason'] == 'stop', d['choices'][0]['finish_reason']
print('SMOKE OK')


In [ ]:
import json
import urllib.request

CASES = {
    'ramble': 'You must register your vehicle at the DMV within 180 days of the effective date on your insurance ID card.',
    'clean': 'Request refund details: purchase details, reason for refund, and relevant documentation.',
}
BASE = (
    'You expand a telegraphic support-draft into one fluent customer reply.\n'
    'Rules:\n'
    '1. Say exactly what the draft says.\n'
    '2. Copy every number, date, URL, name exactly.\n'
    '3. Start directly with content, no greeting or sign-off.\n'
    '4. Output only the reply text.\n'
    'DRAFT:\n{draft}\nREPLY:'
)


def chat(msgs, temp, max_tokens=256):
    body = json.dumps({'messages': msgs, 'temperature': temp,
                       'max_tokens': max_tokens}).encode()
    req = urllib.request.Request(
        f'http://127.0.0.1:{PORT}/v1/chat/completions',
        data=body, headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=600) as r:
        return json.load(r)


results = []
for name, draft in CASES.items():
    prompt = BASE.replace('{draft}', draft)
    prefill = ' '.join(draft.split()[:4]) + ' '
    variants = {
        'nothink': ([{'role': 'user', 'content': prompt + ' /no_think'}], 0),
        'prefill': ([{'role': 'user', 'content': prompt},
                     {'role': 'assistant', 'content': prefill}], 0),
        'temp03': ([{'role': 'user', 'content': prompt}], 0.3),
    }
    for vname, (msgs, temp) in variants.items():
        d = chat(msgs, temp)
        m = d['choices'][0]
        c = m['message']['content'] or ''
        results.append({'case': name, 'variant': vname,
                        'finish': m.get('finish_reason'),
                        'len': len(c), 'head': c[:400]})
        print(name, vname, m.get('finish_reason'), len(c),
              repr(c[:100]), flush=True)
json.dump(results, open('/kaggle/working/probes.json', 'w'), indent=1)
print('saved /kaggle/working/probes.json')
